In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv('../data/cleaned_jobs_salary.csv')

le = LabelEncoder()
cat_cols = ['job_title', 'education_level', 'industry',
            'company_size', 'location', 'remote_work']

for col in cat_cols:
    df[col] = le.fit_transform(df[col])

X = df.drop(columns=['salary'])
y = df['salary']

print(X.columns.tolist())  

['job_title', 'experience_years', 'education_level', 'skills_count', 'industry', 'company_size', 'location', 'remote_work', 'certifications']


In [2]:
#exploring different models
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state = 42)

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=50, n_jobs=-1, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=50, random_state=42)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    print(f"{name}")
    print(f"  MAE : ${mean_absolute_error(y_test, preds):,.0f}")
    print(f"  R-squared  : {r2_score(y_test, preds):.4f}\n")
#Random Forest gave the best R Squared

Linear Regression
  MAE : $21,741
  R-squared  : 0.4560

Random Forest
  MAE : $5,231
  R-squared  : 0.9685

Gradient Boosting
  MAE : $12,130
  R-squared  : 0.8374



In [3]:
from sklearn.model_selection import RandomizedSearchCV

X_tune, _, y_tune, _ = train_test_split(X_train, y_train, train_size = 0.2, random_state= 42)
params = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

grid = RandomizedSearchCV(
    RandomForestRegressor(n_jobs=-1),
    params,
    n_iter=10,      # only tries 10 random combos instead of all 18
    cv=3,           # 3 folds instead of 5
    scoring='r2',
    random_state=42
)
grid.fit(X_tune, y_tune)
print("Best params:", grid.best_params_)

Best params: {'n_estimators': 200, 'min_samples_split': 2, 'max_depth': None}


In [24]:
# Build final model with best params and reduce some estimators to reduce model size to < 100 MB
best_model = RandomForestRegressor(
    n_estimators=50,
    min_samples_split=2,
    max_depth=18,
    n_jobs=-1,
    random_state=42
)

best_model.fit(X_train, y_train)
preds = best_model.predict(X_test)

# Evaluate it
from sklearn.metrics import mean_absolute_error, r2_score

print(f"MAE : ${mean_absolute_error(y_test, preds):,.0f}")
print(f"R²  : {r2_score(y_test, preds):.4f}")

MAE : $5,291
R²  : 0.9678


In [25]:
# Check if model is memorizing training data (overfitting)
train_preds = best_model.predict(X_train)

print(f"Train R²  : {r2_score(y_train, train_preds):.4f}")
print(f"Test  R²  : {r2_score(y_test, preds):.4f}")

Train R²  : 0.9908
Test  R²  : 0.9678


In [26]:
import joblib
from sklearn.preprocessing import LabelEncoder
import pandas as pd

df = pd.read_csv('../data/cleaned_jobs_salary.csv')

cat_cols = ['job_title', 'education_level', 'industry',
            'company_size', 'location', 'remote_work']

encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    le.fit(df[col])
    encoders[col] = le

joblib.dump(best_model, 'salary_model.pkl', compress=3)
joblib.dump(encoders, 'encoders.pkl')

print("Saved columns:", list(encoders.keys()))
print("Model expects:", best_model.feature_names_in_.tolist())

Saved columns: ['job_title', 'education_level', 'industry', 'company_size', 'location', 'remote_work']
Model expects: ['job_title', 'experience_years', 'education_level', 'skills_count', 'industry', 'company_size', 'location', 'remote_work', 'certifications']
